# 📞 4 モデル伝言ゲーム — 文体の変身チェーン

**遊びの実験**: 1 つの日本語文を、4 モデルが順番に **異なる文体** に変換していく伝言ゲーム。各モデルは「自分の前のモデルが出した文」しか見ない (元の文は見ない)。

変換チェーン:
1. 🌸 LLM-jp 8b  → **超ていねい敬語** に変換
2. 🗻 LLM-jp 32b → **ヤンキー / 関西弁** に変換
3. 🐉 Qwen 27b   → **古文 (擬古文)** に変換
4. 💎 Gemma 31b  → **5-7-5 の俳句** に圧縮

**観察ポイント**:
- 🌀 4 段階を経て意味はどれくらい残るか / 飛ぶか
- 🎨 各文体の「衣装」を 4 モデルがどう着こなすか
- 🔁 最後に「元の意味を推定して」と頼んで、復元率を見る


## 1. セットアップ

In [ ]:
import os, re, random
from openai import OpenAI
from IPython.display import display, Markdown

client = OpenAI(
    base_url="https://llm-jp-playground.apps.llmc.nii.ac.jp/api/v1",
    api_key=os.environ.get("LLMJP_API_KEY", "dummy"),
    timeout=300.0,
)

def chat(model, prompt, system="日本語で。", max_tokens=3000, temperature=0.8):
    """内部 streaming 1 ショット。thinking モデルでも reasoning を落とさない。"""
    sys_msg = system + "\n\n/no_think"
    msgs = [{"role": "system", "content": sys_msg},
            {"role": "user", "content": prompt}]
    extra = {"chat_template_kwargs": {"enable_thinking": False}}
    def _try(use_extra):
        kwargs = dict(model=model, messages=msgs, max_tokens=max_tokens,
                      temperature=temperature, stream=True)
        if use_extra:
            kwargs["extra_body"] = extra
        return client.chat.completions.create(**kwargs)
    try:
        stream = _try(True)
    except Exception:
        stream = _try(False)
    content, reasoning = [], []
    for chunk in stream:
        if not chunk.choices: continue
        d = chunk.choices[0].delta
        c = getattr(d, "content", None)
        if c: content.append(c)
        for f in ("reasoning_content", "reasoning"):
            v = getattr(d, f, None)
            if v: reasoning.append(v); break
    text = "".join(content).strip()
    return text if text else "".join(reasoning).strip()

ALL = [m.id for m in client.models.list().data]
def pick(s):
    for m in ALL:
        if s.lower() in m.lower(): return m
    raise RuntimeError(f"no model matching {s!r}")

VOICES = {
    "🌸 LLM-jp 8b":  pick("llm-jp-4-8b"),
    "🗻 LLM-jp 32b": pick("llm-jp-4-32b"),
    "🐉 Qwen 27b":   pick("qwen"),
    "💎 Gemma 31b":  pick("gemma"),
}
print("4 モデル準備完了:")
for n, m in VOICES.items():
    print(f"  {n:18s} → {m}")


## 2. 元の文を決めて変身チェーン開始

In [ ]:
ORIGINAL = (
    "近代日本の数学者は、西洋の論文を翻訳しつつ、独自の研究分野も切り開いた。"
)

# (voice_name, model_id, transformation instruction)
stages = [
    (
        "🌸 LLM-jp 8b",  pick("llm-jp-4-8b"),
        "以下の文を、最大級の **超ていねい敬語** に書き換えてください。出力は変換後の文 1 つだけ、説明不要。"
    ),
    (
        "🗻 LLM-jp 32b", pick("llm-jp-4-32b"),
        "以下の文を、**ヤンキー / 関西弁の口調** に書き換えてください。意味は出来るだけ保ったまま。出力は変換後の文 1 つだけ、説明不要。"
    ),
    (
        "🐉 Qwen 27b",   pick("qwen"),
        "以下の文を、**江戸期の擬古文 (候文・漢文訓読体)** に書き換えてください。出力は変換後の文 1 つだけ、説明不要。"
    ),
    (
        "💎 Gemma 31b",  pick("gemma"),
        "以下の文の核となる意味を、**5-7-5 の俳句 1 句** に圧縮してください。出力は俳句 1 行のみ、説明不要。"
    ),
]

def extract_first_line(raw):
    lines = [l.strip(" 　\t-・*「」『』\"\'") for l in raw.strip().splitlines()]
    lines = [l for l in lines if l and not l.startswith("#")]
    candidates = [l for l in lines if len(l) >= 4]
    return (candidates[0] if candidates else "(no output)")

current = ORIGINAL
chain = [("📄 原文", current)]
print(f"📄 原文:\n  {current}\n")
for voice, model, instr in stages:
    pmt = f"{instr}\n\n対象の文: {current}"
    raw = chat(model, pmt, temperature=0.8, max_tokens=2500)
    out = extract_first_line(raw)
    chain.append((voice, out))
    print(f"{voice}:\n  {out}\n")
    current = out


## 3. 変遷をきれいに表示

In [ ]:
body = "\n\n".join(
    f"**{stage}** ─\n  > {text}" for stage, text in chain
)
display(Markdown(f"## 🔄 文体変身チェーン\n\n{body}"))


## 4. 復元テスト — 最終形から元の意味を当てられるか

In [ ]:
# 第 5 のモデル (LLM-jp 32b) に最終形のみを見せて、元の意味を推定させる
final = chain[-1][1]
reconstruct_prompt = (
    "以下は、ある日本語の文を 4 段階で変身させた末の最終形 (俳句) です。\n"
    "この句から、**元のもとになった事実関係**を 1 文で復元してみてください。\n"
    "出力は復元文 1 つだけ、説明不要。\n\n"
    f"最終形: {final}"
)
recon_name, recon_model = ("🗻 LLM-jp 32b", pick("llm-jp-4-32b"))
reconstructed = extract_first_line(chat(recon_model, reconstruct_prompt,
                                        temperature=0.3, max_tokens=2000))

display(Markdown(
    f"### 🔍 復元実験 ({recon_name} が最終形だけを見て推定)\n\n"
    f"- **原文**: {ORIGINAL}\n"
    f"- **最終形**: {final}\n"
    f"- **{recon_name} の復元**: {reconstructed}\n\n"
    "見比べてみると、伝言ゲームでどこまで意味が残っているか / 何が削ぎ落とされたかが分かります。"
))


## おまけ

- `ORIGINAL` を変えて再実行 — 学術文 / ニュース文 / 詩 / 日常会話 などジャンルで効果が違って面白い
- `stages` の文体指定を入れ替えると別の楽しみ方ができます (例: 漫才口調 / 哲学的独白 / 江戸戯作風)
- 教訓: **意味の核は俳句 17 音にも宿る** (場合がある) ことを観察できれば、要約系タスクの感覚も掴めます
